# SimpleQueuePredictor Json to Json

In [1]:
import pandas as pd
import json
from pandas.io.json import json_normalize
import datetime
from statsmodels.tsa.ar_model import AR

Open the file and read content to a list

In [3]:
with open('20200101-queues-sample.json', 'r') as f:
    res = list()
    for line in f.readlines():
        res.append(json.loads(line))

Build a pandas dataframe from list

In [4]:
df = pd.DataFrame(res)

Narrow it down to the _source column

In [6]:
df = df['_source']

Normalize Dataframe

In [7]:
df = json_normalize(df)

Narrow it down to the products queue

In [9]:
df = df[df['name'] == "products"]

Set the timestamp date as index

In [10]:
df.index = df["timestamp"]

Convert the index to datetimeformat

In [11]:
df.index = pd.to_datetime(df.index, format='%Y-%m-%dT%H:%M:%S.%f%z').sort_values()

Drop unnecessary columns

In [12]:
df.drop(columns=['timestamp', 'name', 'tier', 'items', 'querytime'], inplace=True)

Resample Data to 1min Interval 

In [13]:
df = df.resample('1T').mean()

Fill all NA values with 0 

In [14]:
df = df.fillna(0)

## ML Model

#### Train/Test Split

In [16]:
pred_horizon = 20 #minutes

In [17]:
data = df['size']

#### AR

In [18]:
model_ar = AR(data)
model_ar_fit = model_ar.fit(maxlag= 10,ic='bic', trend='nc', method='cmle', maxiter=20)

In [19]:
pred_test = model_ar_fit.predict(start=len(data), end=len(data)+pred_horizon-1, dynamic=True)

Build Prediction Dataframe

In [20]:
time_stamps = pd.date_range(start=data.index[-1]+datetime.timedelta(minutes=1), periods=20, freq='T',tz='utc')

In [21]:
d = {'timestamp': time_stamps, 'size': pred_test}
pred_df = pd.DataFrame(data=d)

In [22]:
pred_df['timestamp'] = pred_df.timestamp.map(lambda x: datetime.datetime.strftime(x, '%Y-%m-%dT%H:%M:%S.%f%z'))

In [23]:
pred_df

,timestamp,size
2019-12-30 03:20:00+00:00,2019-12-30T03:20:00.000000+0000,33204.994571
2019-12-30 03:21:00+00:00,2019-12-30T03:21:00.000000+0000,33077.480708
2019-12-30 03:22:00+00:00,2019-12-30T03:22:00.000000+0000,32950.456524
2019-12-30 03:23:00+00:00,2019-12-30T03:23:00.000000+0000,32823.920139
2019-12-30 03:24:00+00:00,2019-12-30T03:24:00.000000+0000,32697.869679
2019-12-30 03:25:00+00:00,2019-12-30T03:25:00.000000+0000,32572.303277
2019-12-30 03:26:00+00:00,2019-12-30T03:26:00.000000+0000,32447.219077
2019-12-30 03:27:00+00:00,2019-12-30T03:27:00.000000+0000,32322.615224
2019-12-30 03:28:00+00:00,2019-12-30T03:28:00.000000+0000,32198.489876
2019-12-30 03:29:00+00:00,2019-12-30T03:29:00.000000+0000,32074.841194


Build list in json format with the prediction Dataframe

In [339]:
final = []
for index, row in pred_df.iterrows():
    doc_data = {'_index': 'queues',
                 '_type': '_doc',
                  #'_id': 'random',
               #'_score': 1,
               '_source': {
                            'timestamp': row['timestamp'],
                                'tier' : 'pic',
                                'name' : 'products',
                     #     'querytime' : 0,
                                'size' : row['size'],
                         #     'items' : " ".join(items)
                         }}
    final.append(doc_data)

Write the elements of the list as json file

In [341]:
with open('prediction.json', 'w') as f:
    f.writelines([json.dumps(elem) for elem in final])